# Cancelamento de reservas de hotel: qualidade dos dados do teste

**Pergunta:** os dados de 2017 (o teste) têm algum problema ou mudança que o modelo precisa considerar?

**Este notebook** confere as reservas de 2017 antes da avaliação do modelo: se a limpeza pesa igual, se a qualidade é a mesma dos anos de treino, se há categorias novas e o que muda no perfil das reservas. **Só as variáveis entram aqui, nunca o cancelamento de 2017:** a taxa do teste fica para a avaliação, no `05_modelo.ipynb`. Nenhuma regra de limpeza ou de variável muda neste notebook; elas vêm do `01_exploratoria.ipynb` e do `02_variaveis.ipynb`.

## Resumo

- **Os dados de 2017 não têm problema novo de qualidade:** a limpeza tira 0,49% do escopo (contra cerca de 1% no treino), não há categorias novas, e as variáveis montadas não têm vazios.
- **O perfil muda:** mais pedidos especiais (+8,7 pontos), mais agência online e grupos (+8,5), mais antecedência acima de 25 dias (+5,7) e diárias mais altas (mediana de 95 € para 108 €), sempre de janeiro a agosto.
- **Consequência:** o `05_modelo.ipynb` confere a calibração em 2017 no total, por segmento e no efeito do pedido especial.

In [1]:
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append('..')   # o módulo de funções do projeto fica em analise/
from funcoes import usar_tabelas_simples, preparar_base, montar_variaveis, SITUACAO_VARIAVEIS, NOMES_VARIAVEIS

usar_tabelas_simples()   # todas as tabelas: nomes numa linha, índice como coluna, 4 casas
pd.set_option('display.max_rows', 100)

hotel = pd.read_csv('../../dados/hotel_bookings.csv')
base, _ = preparar_base(hotel)
jan_a_ago = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August']

## 1. A limpeza em 2017

#### Reservas que saem em cada passo, por ano

A função `preparar_base` aplicada a cada ano separado: o escopo (só reservas sem depósito) e as remoções do notebook 01, seção 4. A última linha é a parcela das reservas do escopo que sai na limpeza.

In [2]:
# quantas reservas saem em cada passo, em cada ano
passos = {ano: preparar_base(hotel[hotel['arrival_date_year'] == ano])[1] for ano in (2015, 2016, 2017)}
tabela = pd.DataFrame({ano: p['saem neste passo'] for ano, p in passos.items()}).drop(index='base inteira').fillna(0) + 0.0   # + 0.0 evita o -0 da subtração
tabela.loc['ficam'] = [p['reservas'].iloc[-1] for p in passos.values()]
escopo = pd.Series({ano: p.loc['sem depósito (escopo)', 'reservas'] for ano, p in passos.items()})
tabela.loc['% do escopo que sai na limpeza'] = (escopo - tabela.loc['ficam']) / escopo * 100
tabela.rename_axis('passo').round(2)

passo,2015,2016,2017
sem depósito (escopo),3671,6925,4153
children preenchido,4,0,0
com hóspede,29,84,67
diária entre 0 e 1.000,0,0,1
com pelo menos uma noite,147,388,110
ficam,18145,49310,36356
% do escopo que sai na limpeza,0.9800,0.9500,0.4900


**Leitura.**

- **A limpeza tira 0,49% do escopo em 2017, contra 0,98% em 2015 e 0,95% em 2016.** A diferença vem quase toda das reservas com zero noites: 110 em 2017, contra 147 e 388.
- **Nenhuma regra pesa fora do comum em 2017.** Ficam 36.356 reservas no teste.

## 2. Qualidade, ano a ano

#### Indicadores de qualidade

A parcela de reservas em cada situação que o notebook 01 examinou, na base já limpa. 2015 só tem julho a dezembro, e 2017, janeiro a agosto: a coluna de 2016 de janeiro a agosto compara 2017 com os mesmos meses.

In [3]:
# indicadores de qualidade por ano (base limpa)
indicadores = pd.DataFrame({
    'sem agência': base['agent'].isna(),
    'paga por empresa': base['company'].notna(),
    'linha com gêmea (blocos)': base['tem_gemea'],
    'diária zero': base['adr'] == 0,
    'diária acima de 240 € (p99 do treino)': base['adr'] > 240,
    'antecedência acima de 365 dias': base['lead_time'] > 365,
    'mais de 14 noites (p99 do treino)': base['noites'] > 14,
    'sem adulto': base['adults'] == 0,
    'meal = Undefined': base['meal'] == 'Undefined',
    'país vazio': base['country'].isna(),
})
ano = base['arrival_date_year']
jan_ago = base['arrival_date_month'].isin(jan_a_ago)
pd.DataFrame({
    '2015 (jul–dez)': indicadores[ano == 2015].mean(),
    '2016': indicadores[ano == 2016].mean(),
    '2016 (jan–ago)': indicadores[(ano == 2016) & jan_ago].mean(),
    '2017 (jan–ago)': indicadores[ano == 2017].mean(),
}).rename_axis('indicador').round(4)

indicador,2015 (jul–dez),2016,2016 (jan–ago),2017 (jan–ago)
sem agência,0.1505,0.1383,0.1309,0.1199
paga por empresa,0.0635,0.0588,0.0547,0.0562
linha com gêmea (blocos),0.3731,0.2274,0.2132,0.2114
diária zero,0.0227,0.0095,0.0093,0.0079
diária acima de 240 € (p99 do treino),0.0045,0.0116,0.0145,0.0286
antecedência acima de 365 dias,0.0006,0.0103,0.0020,0.0208
mais de 14 noites (p99 do treino),0.0037,0.0033,0.0035,0.0049
sem adulto,0.0008,0.0029,0.0031,0.0017
meal = Undefined,0.0094,0.0096,0.0099,0.0102
país vazio,0.0066,0.0053,0.0055,0.0022


**Leitura.**

- **Nos mesmos meses, 2017 se parece com 2016 na maior parte dos indicadores:** sem agência, 12,0% contra 13,1%; linhas com gêmea, 21,1% contra 21,3%; diária zero, 0,8% contra 0,9%; `meal = Undefined`, 1,0% nos dois.
- **Duas caudas crescem:** diárias acima de 240 € passam de 1,5% para 2,9%, e antecedências acima de 365 dias, de 0,2% para 2,1%. Acompanham a alta da diária e da antecedência medianas (seção 4).
- **O ano diferente é 2015:** 37,3% de linhas com gêmea, contra 22,7% em 2016, e 2,3% de diárias zero, contra 1,0%. É o peso dos blocos de quartos em 2015 (02, seção 3.1).
- **Nenhum problema novo de qualidade em 2017.**

## 3. Categorias novas

Valores que aparecem em 2017 e não aparecem no treino deixariam o modelo sem coeficiente para eles. A conferência usa as colunas originais das variáveis categóricas e o resultado da `montar_variaveis` no teste.

In [4]:
# valores de 2017 que o treino não tem, e vazios nas variáveis montadas para o teste
treino_bruto = hotel[hotel['arrival_date_year'] < 2017]
teste_bruto = hotel[hotel['arrival_date_year'] == 2017]
for coluna in ['hotel', 'market_segment', 'distribution_channel', 'arrival_date_month', 'meal', 'customer_type']:
    novos = sorted(set(teste_bruto[coluna].dropna()) - set(treino_bruto[coluna].dropna()))
    print(f"{coluna}: {', '.join(novos) if novos else 'nenhum valor novo'}")
teste = montar_variaveis(base[ano == 2017])
print('Vazios nas variáveis montadas do teste:', int(teste.isna().sum().sum()))

hotel: nenhum valor novo
market_segment: nenhum valor novo
distribution_channel: nenhum valor novo
arrival_date_month: nenhum valor novo
meal: nenhum valor novo


customer_type: nenhum valor novo
Vazios nas variáveis montadas do teste: 0


**Leitura.**

- **2017 não traz nenhum valor novo** nas colunas categóricas, e a `montar_variaveis` monta as variáveis do teste sem nenhum vazio. O modelo tem coeficiente para todas as reservas do teste.

## 4. O perfil das reservas em 2017

As comparações usam janeiro a agosto nos dois anos, para tirar o efeito da época do ano: 2017 só tem esses meses.

#### Medianas

In [5]:
# medianas de janeiro a agosto: 2016 × 2017
mesmos_meses = base[jan_ago & ano.isin([2016, 2017])]
(mesmos_meses.groupby('arrival_date_year')[['lead_time', 'noites', 'adr']].median().T
 .set_axis(['mediana_2016', 'mediana_2017'], axis=1).rename_axis('variavel'))

variavel,mediana_2016,mediana_2017
lead_time,55,73
noites,3,3
adr,95,108


**Leitura.**

- **Em 2017, as reservas são feitas com mais antecedência e pagam mais caro.** Nos mesmos meses, a antecedência mediana sobe de 55 para 73 dias, e a diária mediana, de 95 € para 108 €. O total de noites não muda (3).

#### Níveis das variáveis

A parcela de cada nível das 13 variáveis que vão para o stepwise (o canal saiu no `03_variaveis_em_conjunto.ipynb`), de janeiro a agosto de 2016 e de 2017, e a diferença em pontos percentuais.

In [6]:
# parcela de cada nível, janeiro a agosto de 2016 × 2017
variaveis = montar_variaveis(mesmos_meses)
partes = {}
for v in [v for v in SITUACAO_VARIAVEIS if v != 'canal']:
    t = pd.crosstab(variaveis[v], variaveis['arrival_date_year'], normalize='columns')
    t.columns = ['pct_2016', 'pct_2017']
    t['dif_pp'] = (t['pct_2017'] - t['pct_2016']) * 100
    partes[NOMES_VARIAVEIS[v]] = t
pd.concat(partes, names=['variavel', 'nivel']).round(4)

variavel,nivel,pct_2016,pct_2017,dif_pp
antecedência,mais de 25,0.6513,0.7088,5.7486
antecedência,0,0.0648,0.0474,-1.7386
antecedência,1–8,0.1559,0.1181,-3.7866
antecedência,9–25,0.1280,0.1257,-0.2234
noites,2–3,0.4196,0.4088,-1.0857
noites,1,0.2107,0.1733,-3.7410
noites,4 ou mais,0.3697,0.4180,4.8268
diária,mais de 59,0.8558,0.8927,3.6905
diária,até 59,0.1442,0.1073,-3.6905
sozinho,não,0.8117,0.8218,1.0124


**Leitura.**

- **As maiores mudanças:** quem faz pedido especial passa de 44,2% para 52,9% (+8,7 pontos); agência online e grupos, de 61,9% para 70,5% (+8,5); mais de 25 dias de antecedência, de 65,1% para 70,9% (+5,7); 4 noites ou mais, de 37,0% para 41,8% (+4,8). Diárias até 59 € caem de 14,4% para 10,7% (−3,7).
- **As outras variáveis mudam até 1,5 ponto.**
- **As faixas em euros envelhecem:** com a diária subindo, a faixa até 59 € perde reservas a cada ano.

#### Segmento por dentro

O segmento do modelo junta agência online e grupos. A tabela mostra cada segmento original, de janeiro a agosto de 2016 e de 2017.

In [7]:
# segmentos originais, janeiro a agosto de 2016 × 2017
t = pd.crosstab(mesmos_meses['market_segment'], mesmos_meses['arrival_date_year'], normalize='columns')
t.columns = ['pct_2016', 'pct_2017']
t['dif_pp'] = (t['pct_2017'] - t['pct_2016']) * 100
t.sort_values('pct_2017', ascending=False).rename_axis('segmento').round(4)

segmento,pct_2016,pct_2017,dif_pp
Online TA,0.5515,0.6205,6.8956
Direct,0.1192,0.1258,0.6590
Offline TA/TO,0.2015,0.1236,-7.7938
Groups,0.0678,0.0840,1.6222
Corporate,0.0517,0.0375,-1.4260
Complementary,0.0061,0.0057,-0.0424
Aviation,0.0021,0.0029,0.0854


**Leitura.**

- **Dentro do segmento "agência online e grupos", a mudança vem da agência online:** de 55,2% para 62,1% (+6,9 pontos). Grupos sobem menos, de 6,8% para 8,4%.
- **A operadora tradicional cai de 20,2% para 12,4% (−7,8 pontos),** quase toda a queda do segmento "demais".

## 5. O que isso muda para o modelo

- **Os dados de 2017 estão prontos para a avaliação.** A limpeza pesa menos que no treino, a qualidade é a mesma de 2016, não há categorias novas, e as variáveis do teste não têm vazios.
- **O perfil das reservas muda em 2017, em dois sentidos.** Crescem perfis que no treino cancelam mais: antecedência acima de 25 dias, agência online e grupos, 4 noites ou mais e diárias acima de 59 € (02, seção 4). E cresce um perfil que cancela menos: quem faz pedido especial.
- **O modelo capta a parte da mudança que passa por essas variáveis,** porque elas estão nele. O que não passa por elas aparece na avaliação do `05_modelo.ipynb`, como diferença entre a probabilidade prevista e a taxa real.
- **Para o 05:** conferir a calibração em 2017 no total e por segmento, e conferir se o efeito do pedido especial se mantém, já que a parcela de pedidos foi a que mais mudou.